## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.sparse as sps
import gc
import glob

from Challenge.paths import XGBOOST_DATAFRAMES

Running on local — storage at: /home/luigi/RecSys


## **Merge fold dataframes**

In [4]:
# Path where your current script saved the files
INPUT_DIR = os.path.join(XGBOOST_DATAFRAMES, "training_data")
INPUT_FILES = sorted(glob.glob(os.path.join(INPUT_DIR, "fold_*.parquet")))

# Path to save the XGBoost-ready files
OUTPUT_DIR = os.path.join(XGBOOST_DATAFRAMES, "ready_for_xgboost")
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_BINS = 10  # 4 mil rows
GROUP_COL = 'UserID' # Ensure this matches your column name

print(f"Found {len(INPUT_FILES)} input files.")

Found 5 input files.


In [5]:
print("--- PHASE 1: Pre-Computation Safety Check ---")
# Calculate total input rows to compare against later
# Reading only one column is very RAM efficient
total_input_rows = 0
print("Counting input rows...")
for f in INPUT_FILES:
    # We only load the Group column to save RAM
    _df = pd.read_parquet(f, columns=[GROUP_COL])
    total_input_rows += len(_df)
    del _df
    gc.collect()

print(f"Total Input Rows Detected: {total_input_rows:,}")
print("---------------------------------------------")

total_output_rows = 0

for bin_idx in range(NUM_BINS):
    print(f"\n--- Building Clean Partition {bin_idx} ---")
    bin_chunks = []
    
    for fpath in INPUT_FILES:
        # Load file
        df = pd.read_parquet(fpath)
        
        # Filter for users belonging to this specific bin
        mask = (df[GROUP_COL] % NUM_BINS) == bin_idx
        chunk = df[mask].copy()
        
        if not chunk.empty:
            bin_chunks.append(chunk)
            
        del df, mask, chunk
        gc.collect()
    
    # Merge and Save
    if bin_chunks:
        print(f"Merging partition {bin_idx}...")
        df_clean = pd.concat(bin_chunks)
        
        # 1. CRITICAL OPERATION: SORT
        df_clean = df_clean.sort_values(GROUP_COL)
        
        # ==========================================
        # SAFETY CHECK 1: BIN CONSISTENCY
        # ==========================================
        # Verify that every ID in this dataframe actually belongs here
        # If this fails, your modulo logic is broken.
        check_mask = (df_clean[GROUP_COL] % NUM_BINS) == bin_idx
        if not check_mask.all():
            bad_ids = df_clean.loc[~check_mask, GROUP_COL].unique()
            raise ValueError(f"CRITICAL ERROR: Found UserIDs in Bin {bin_idx} that do not belong! Examples: {bad_ids[:5]}")

        # ==========================================
        # SAFETY CHECK 2: SORT ORDER
        # ==========================================
        # Verify IDs are monotonic (0,0,1,1,1,5,5...). 
        # If this is False, XGBoost will silently fail.
        if not df_clean[GROUP_COL].is_monotonic_increasing:
             raise ValueError(f"CRITICAL ERROR: Partition {bin_idx} is not sorted by {GROUP_COL}!")

        # Update counter
        total_output_rows += len(df_clean)
        
        # Save
        save_path = os.path.join(OUTPUT_DIR, f"clean_part_{bin_idx}.parquet")
        df_clean.to_parquet(save_path, index=False)
        print(f"Saved {save_path} ({len(df_clean):,} rows) [Checks Passed]")
        
        del df_clean, bin_chunks, check_mask
        gc.collect()
    else:
        print(f"Warning: Bin {bin_idx} is empty.")

print("\n--- PHASE 3: Final Volume Check ---")
print(f"Input Rows:  {total_input_rows:,}")
print(f"Output Rows: {total_output_rows:,}")

if total_input_rows == total_output_rows:
    print("SUCCESS: Data volume matches perfectly. Files are ready for XGBoost.")
else:
    diff = total_input_rows - total_output_rows
    raise ValueError(f"DATA LOSS DETECTED: Lost {diff} rows during shuffling!")

--- PHASE 1: Pre-Computation Safety Check ---
Counting input rows...
Total Input Rows Detected: 41,894,675
---------------------------------------------

--- Building Clean Partition 0 ---
Merging partition 0...
Saved /home/luigi/RecSys/xg_boost_data/dataframes/ready_for_xgboost/clean_part_0.parquet (4,190,859 rows) [Checks Passed]

--- Building Clean Partition 1 ---
Merging partition 1...
Saved /home/luigi/RecSys/xg_boost_data/dataframes/ready_for_xgboost/clean_part_1.parquet (4,185,342 rows) [Checks Passed]

--- Building Clean Partition 2 ---
Merging partition 2...
Saved /home/luigi/RecSys/xg_boost_data/dataframes/ready_for_xgboost/clean_part_2.parquet (4,189,441 rows) [Checks Passed]

--- Building Clean Partition 3 ---
Merging partition 3...
Saved /home/luigi/RecSys/xg_boost_data/dataframes/ready_for_xgboost/clean_part_3.parquet (4,187,034 rows) [Checks Passed]

--- Building Clean Partition 4 ---
Merging partition 4...
Saved /home/luigi/RecSys/xg_boost_data/dataframes/ready_for_xgbo